# QLoRA Fine-tuning: Mistral-7B Grumpy Senior
**Goal**: Обучить Mistral-7B отвечать как грубый синьор, который ворчит, но технически точен.

**Hardware**: Google Colab T4 GPU  
**Method**: QLoRA (Quantized LoRA) — 4-bit quantization + Low-Rank Adaptation  
**Dataset**: 200 пар instruction/response из `grumpy_senior.jsonl`

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

!pip install -q torch==2.4.1 --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers==4.40.2 peft==0.11.1 bitsandbytes==0.43.1 datasets==2.19.0 accelerate==0.30.1 trl==0.9.6
!pip install -q mlflow==2.9.2

import torch, transformers, peft, bitsandbytes, mlflow

print(f"{torch.__version__}, GPU: {torch.cuda.is_available()}")
print(f"protobuf: {__import__('google.protobuf', fromlist=['__version__']).__version__}")
print(f"Transformers: {transformers.__version__}, PEFT: {peft.__version__}")
print(f"MLflow: {mlflow.__version__}")

In [ ]:
mlflow.set_experiment("grumpy-senior-qlora")
mlflow.start_run(run_name="mistral-7b-qlora")

mlflow.log_params({
    "model": "mistralai/Mistral-7B-v0.1",
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 2e-4,
    "num_epochs": 3,
    "batch_size": 2,
    "gradient_accumulation_steps": 4,
    "quantization": "4-bit NF4",
    "dataset_size": 200,
})

print(f"MLflow run started: {mlflow.active_run().info.run_id}")

In [ ]:
# Тест ГПУ
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import json
import os
from datasets import Dataset
from pathlib import Path


DATASET_PATH = "/content/grumpy_senior.jsonl"
OUTPUT_DIR = "/content/mistral-grumpy-lora"



data = []
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    print(f"Загружено {len(data)} примеров")
else:
    print(f"Файл не найден: {DATASET_PATH}")
    print(f"Доступные файлы в /content:")
    print(os.listdir("/content"))


if data:
    print("\n--- Пример 1 ---")
    print(f"Instruction: {data[0]['instruction']}")
    print(f"Response: {data[0]['response'][:200]}...")

In [ ]:

dataset = Dataset.from_dict({
    'instruction': [item['instruction'] for item in data],
    'response': [item['response'] for item in data]
})


split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
val_dataset = split['test']

print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")

In [ ]:
def formatting_func(examples):
    """Преобразуем instruction/response в text для обучения"""
    output_texts = []

    for i in range(len(examples['instruction'])):
        text = f"""[INST] {examples['instruction'][i]} [/INST]
{examples['response'][i]}</s>"""
        output_texts.append(text)

    return {"text": output_texts}


train_dataset = train_dataset.map(
    formatting_func,
    batched=True,
    remove_columns=['instruction', 'response']
)

val_dataset = val_dataset.map(
    formatting_func,
    batched=True,
    remove_columns=['instruction', 'response']
)

print(f"Dataset formatted")
print(f"\nПример обучающей строки:")
print(train_dataset[0]['text'][:300])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# параметры квантизации
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "mistralai/Mistral-7B-v0.1"
print(f"Loading {model_name}...")


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)


tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {model_name}")
print(f"Total params: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


model = prepare_model_for_kbit_training(model)


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


model = get_peft_model(model, lora_config)


trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"LoRA applied")
print(f"Trainable params: {trainable_params / 1e6:.2f}M")
print(f"Total params: {total_params / 1e9:.2f}B")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    optim="paged_adamw_8bit",
    save_strategy="epoch",
    evaluation_strategy="steps",
    eval_steps=10,
    save_total_limit=2,
    logging_steps=5,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="mlflow",
    run_name="mistral-grumpy-qlora",
    seed=42,
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

print("Training arguments configured")

In [ ]:
def tokenize_func(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_dataset = train_dataset.map(tokenize_func, batched=True, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize_func, batched=True, remove_columns=["text"])

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Колонки: {train_dataset.column_names}")

In [ ]:
from transformers import DataCollatorForSeq2Seq

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=8
    ),
)
print("Trainer initialized")

In [ ]:
# Запускаем обучение
train_result = trainer.train()

print("\n" + "="*50)
print("Training completed!")
print(f"Final loss: {train_result.training_loss:.4f}")
print("="*50)

In [ ]:
mlflow.log_metric("final_train_loss", train_result.training_loss)
mlflow.end_run()
print(f"MLflow run завершён")

In [ ]:
# Сохраняем только адаптер
adapter_dir = f"{OUTPUT_DIR}/adapter"
model.save_pretrained(adapter_dir)


tokenizer.save_pretrained(adapter_dir)

print(f"LoRA adapter saved to: {adapter_dir}")
print(f"\nМожно загрузить так:")
print(f"""
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained(
    \"{adapter_dir}\",
    device_map=\"auto\"
)
tokenizer = AutoTokenizer.from_pretrained(\"{adapter_dir}\")
""")

In [ ]:
# Проверяем размер адаптера
import os
from pathlib import Path

adapter_size = sum(f.stat().st_size for f in Path(adapter_dir).rglob('*') if f.is_file()) / 1e6
print(f"Adapter size: {adapter_size:.1f} MB")
print(f"\nСохранённые файлы:")
for f in sorted(os.listdir(adapter_dir)):
    path = os.path.join(adapter_dir, f)
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1e6
        print(f"  {f}: {size:.2f} MB")

In [ ]:

model.eval()

# Тестовые вопросы
test_prompts = [
    "Как настроить Docker контейнер?",
    "Объясни SQL join",
    "Что такое async/await в JavaScript?"
]

print("Testing fine-tuned model:")
print("="*60)

for prompt in test_prompts:
    input_text = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)

    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()

    print(f"\nQ: {prompt}")
    print(f"A: {response[:200]}...\n")
    print("-"*60)

In [ ]:

print("\n" + "="*60)
print("Fine-tuning complete!")
print("="*60)
print(f"\nРезультаты:")
print(f"  • Модель: Mistral-7B-v0.1")
print(f"  • Метод: QLoRA (4-bit quantization)")
print(f"  • Примеров: {len(train_dataset)} train, {len(val_dataset)} val")
print(f"  • Адаптер сохранён в: {adapter_dir}")
print(f"  • Размер адаптера: {adapter_size:.1f} MB")
print(f"\nДля загрузки в production:")
print(f"  1. Скачай содержимое {adapter_dir}")
print(f"  2. Используй AutoPeftModelForCausalLM.from_pretrained()")


In [ ]:
!mlflow ui --port 5000 &
import time
time.sleep(2)
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(5000)"))

In [ ]:
import matplotlib.pyplot as plt
import mlflow

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("grumpy-senior-qlora")

run_id = "c0350fa973334b11986e5fbc65a396e1"


train_loss_history = client.get_metric_history(run_id, "loss")
val_loss_history = client.get_metric_history(run_id, "eval_loss")

train_steps = [m.step for m in train_loss_history]
train_values = [m.value for m in train_loss_history]

val_steps = [m.step for m in val_loss_history]
val_values = [m.value for m in val_loss_history]


plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_values, label='Train Loss', marker='o', color='blue')
plt.plot(val_steps, val_values, label='Val Loss', marker='o', color='orange')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('QLoRA Fine-tuning: Mistral-7B Grumpy Senior')
plt.legend()
plt.grid(True)
plt.tight_layout()


plt.savefig('/content/loss_curve.png', dpi=150)
mlflow.log_artifact('/content/loss_curve.png')
plt.show()
print("График сохранён и залогирован в MLflow")